In [ ]:
import time
import enum
import math
import numpy as np
from Arm_Lib import Arm_Device
import cv2
import ipywidgets.widgets as widgets


######################################################################################################################################################################
######################################################################################################################################################################
#Basic definitions for common vectors (3x1) and rotation matrices (3x3)
ez = np.matrix([[0],[0],[1]])
ey = np.matrix([[0],[1],[0]])
ex = np.matrix([[1],[0],[0]])
e3 = np.matrix([[0],[0],[0]])

#Computes rotation matrices (given degrees)
#returns 3x3 np.matrix
def rotz3(degrees):
    degrees *= 3.14159265/180
    c = math.cos(degrees)
    s = math.sin(degrees)
    return np.matrix([[c,-s,0],[s,c,0], [0,0,1]])
def rotx3(degrees):
    degrees *= 3.14159265/180
    c = math.cos(degrees)
    s = math.sin(degrees)
    return np.matrix([[1,0,0], [0,c,-s], [0,s,c,]])
def roty3(degrees):
    degrees *= 3.14159265/180
    c = math.cos(degrees)
    s = math.sin(degrees)
    return np.matrix([[c,0,s], [0,1,0], [-s,0,c]])

#Computes degrees to radians for a given list of angles
#returns list of len(L) of floats
def degreesToRadians(L):
    L = L.copy()
    for i in range(len(L)):
        L[i] *= 3.1415926535/180.0
    return L
def radiansToDegrees(L):
    L = L.copy()
    for i in range(len(L)):
        L[i] *= 180.0/3.1415926535
    return L

#Computes v^x
#returns: 3x3 matrix
def toCross(mat):
    k1 = mat.getA()[0][0]
    k2 = mat.getA()[1][0]
    k3 = mat.getA()[2][0]
    return np.matrix([[0,-k3, k2],[k3,0,-k1],[-k2,k1,0]])

#Computes magnitude of for n by m matrix (returns sqrt(sum(x_ij)) for all ij). Computes like & for vector magnitude, but can handle any 2d matrix
#returns: float
def getMagnitude(mat):
    a = mat.getA()
    tot = 0
    for i in range(len(a)):
        for j in range(len(a[i])):
            tot += a[i][j]**2
    return math.sqrt(tot)

def normalize(mat):
    a = mat.getA()
    tot = 0
    for i in range(len(a)):
        for j in range(len(a[i])):
            tot += a[i][j]**2
    mag = math.sqrt(tot)
    for i in range(len(a)):
        for j in range(len(a[i])):
            a[i][j] /= mag
    return np.matrix(a)
    




######################################################################################################################################################################
######################################################################################################################################################################
#Computes forward kinematics for dotbof robot
#returns: [ position (3x1 np.matrix), rotation (3x3 np.matrix) ] 
def dofbotFK(jointAngles = [0,0,0,0,0]):
    q = jointAngles

    r01 = rotz3(q[0])
    r12 = roty3(-q[1])
    r23 = roty3(-q[2])
    r34 = roty3(-q[3])
    r45 = rotx3(-q[4])

    l = [61, 43.5, 82.85, 82.85, 73.85, 54.57]
    #p = [ez*(l[0]+l[1]), e3, ex*l[2], -ez*l[3], e3, -ex*(l[4]+l[5]) ]
    p01 = ez*(l[0]+l[1])
    p12 = e3
    p23 = ex*l[2]
    p34 = -ez*l[3]
    p45 = e3
    p5T = -ex*(l[4]+l[5])

    #           Position                                                Rotation
    return [ p01 + r01*(p12+r12*(p23+r23*(p34+r34*(p45+r45*p5T)))),    r01*r12*r23*r34*r45 ]

#Computes jacobian for Dofbot @ jointAngles
#returns: Jacobian (6x5 matrix)
def dofbotJacobian(jointAngles = []):
    l = [61, 43.5, 82.85, 82.85, 73.85, 54.57]
    q = jointAngles

    #Find rotation matrices
    r01 = rotz3(q[0])
    r12 = roty3(-q[1])
    r23 = roty3(-q[2])
    r34 = roty3(-q[3])
    r45 = rotx3(-q[4])

    r02 = r01*r12
    r03 = r02*r23
    r04 = r03*r34
    r05 = r04*r45

    h1 = ez
    h2 = -ey
    h3 = -ey
    h4 = -ey
    h5 = -ex

    #vp01 = ez*(l[0]+l[1])
    vp12 = e3
    vp23 = ex*l[2]
    vp34 = -ez*l[3]
    vp45 = e3
    vp5T = -ex*(l[4]+l[5])

    p1T = vp12+r12*(vp23+r23*(vp34+r34*(vp45+r45*vp5T)))
    p2T = vp23+r23*(vp34+r34*(vp45+r45*vp5T))
    p3T = vp34+r34*(vp45+r45*vp5T)
    p4T = vp45+r45*vp5T
    p5T = vp5T

    a = h1.getA()
    b = (r01*h2).getA()
    c = (r02*h3).getA()
    d = (r03*h4).getA()
    e = (r04*h5).getA()

    f = (toCross(h1)*r01*p1T).getA()
    g = (toCross(r01*h2)*r02*p2T).getA()
    h = (toCross(r02*h3)*r03*p3T).getA()
    i = (toCross(r03*h4)*r04*p4T).getA()
    j = (toCross(r04*h5)*r05*p5T).getA()

    return np.matrix(
        [
            [a[0][0], b[0][0], c[0][0], d[0][0], e[0][0]],
            [a[1][0], b[1][0], c[1][0], d[1][0], e[1][0]],
            [a[2][0], b[2][0], c[2][0], d[2][0], e[2][0]],
            
            [f[0][0], g[0][0], h[0][0], i[0][0], j[0][0]],
            [f[1][0], g[1][0], h[1][0], i[1][0], j[1][0]],
            [f[2][0], g[2][0], h[2][0], i[2][0], j[2][0]]
        ]
    )

#Computes Jacobian, FK Position, and FK Rotation given jointAngles in degrees. 
#returns: [Jacobian, Position, Rotation]
def dofbotJacobianFKPositionFKRotation(jointAngles = []):
    l = [61, 43.5, 82.85, 82.85, 73.85, 54.57]
    q = jointAngles

    #Find rotation matrices
    r01 = rotz3(q[0])
    r12 = roty3(-q[1])
    r23 = roty3(-q[2])
    r34 = roty3(-q[3])
    r45 = rotx3(-q[4])

    r02 = r01*r12
    r03 = r02*r23
    r04 = r03*r34
    r05 = r04*r45

    #define basic axes of rotation
    h1 = ez
    h2 = -ey
    h3 = -ey
    h4 = -ey
    h5 = -ex

    #Position stuff
    vp01 = ez*(l[0]+l[1])
    vp12 = e3
    vp23 = ex*l[2]
    vp34 = -ez*l[3]
    vp45 = e3
    vp5T = -ex*(l[4]+l[5])

    #Compute p1T, P2T, p3T, ... 
    #p1T = vp12+r12*(vp23+r23*(vp34+r34*(vp45+r45*vp5T)))
    #p2T = vp23+r23*(vp34+r34*(vp45+r45*vp5T))
    #p3T = vp34+r34*(vp45+r45*vp5T)
    #p4T = vp45+r45*vp5T
    p5T = vp5T
    p4T = vp45 + r45*p5T
    p3T = vp34 + r34*p4T
    p2T = vp23 + r23*p3T
    p1T = vp12 + r12*p2T

    a = h1.getA()
    b = (r01*h2).getA()
    c = (r02*h3).getA()
    d = (r03*h4).getA()
    e = (r04*h5).getA()

    f = (toCross(h1)*r01*p1T).getA()
    g = (toCross(r01*h2)*r02*p2T).getA()
    h = (toCross(r02*h3)*r03*p3T).getA()
    i = (toCross(r03*h4)*r04*p4T).getA()
    j = (toCross(r04*h5)*r05*p5T).getA()


    jac = np.matrix(
        [
            [a[0][0], b[0][0], c[0][0], d[0][0], e[0][0]],
            [a[1][0], b[1][0], c[1][0], d[1][0], e[1][0]],
            [a[2][0], b[2][0], c[2][0], d[2][0], e[2][0]],
            
            [f[0][0], g[0][0], h[0][0], i[0][0], j[0][0]],
            [f[1][0], g[1][0], h[1][0], i[1][0], j[1][0]],
            [f[2][0], g[2][0], h[2][0], i[2][0], j[2][0]]
        ]
    )

    rot = r01*r12*r23*r34*r45
    pos = vp01 + r01*(vp12+r12*(vp23+r23*(vp34+r34*(vp45+r45*vp5T))))

    return [jac, pos, rot]



def dofbotIK_helper(R):
    #assert np.linalg.norm(R[0:2,0]) > np.finfo(float).eps * 10.0, "Singular rpy requested"
    r=np.arctan2(R[2,1],R[2,2])
    y=np.arctan2(R[1,0], R[0,0])
    p=np.arctan2(-R[2,0], np.linalg.norm(R[2,1:3]))
    return (r,p,y)

#Computes joint angles for a dofbot to go to a desired position and rotatin (Pd and Rd)
#returns List of Floats (length = 5)    (the joint angles in degrees)
def dofbotIK(Pd = np.matrix([[20.58],[11.88],[14.64]]), Rd = rotx3(0), q_initial_degrees = [25,50,75,30,30], maximumTargetDifference = 0.1, alpha = 0.00001, Nmax = 1000):
    
    if type(Pd) == list: #if desired position is a list, convert to a 3r x 1c matrix
        Pd = np.matrix([[Pd[0]],[Pd[1]],[Pd[2]]])
    
    #Set initial rotation
    q = [0]*5 #q IS IN RADIANS
    for i in range(len(q)):
        q[i] = q_initial_degrees[i]*3.1415926535/180.00
    
    #Using Jacobian Transpose method, 
    for i in range(Nmax):
        #Get end effector position, rotation, and Jacobian for current q
        q_degrees = radiansToDegrees(q) #Convert to degrees to use FK functions nad Jacobian
        #P = dofbotFK(q_degrees, False)
        #R = dofbotFK_rot(q_degrees, False)
        #J = dofbotJacobian(q_degrees, False)
        J, P, R = dofbotJacobianFKPositionFKRotation(q_degrees)

        #Compute target error & rotational error, and positionDifference vector
        Er = R * np.matrix.transpose(Rd)
        er = np.matrix( np.array(dofbotIK_helper(Er)) ).transpose()

        #dR = R*np.matrix.transpose(Rd.copy())
        posDif = P - Pd
        rotDif = R - Rd
        if getMagnitude(posDif) < maximumTargetDifference and getMagnitude(rotDif) < maximumTargetDifference:
            print("...close enough...")
            break
        dX = np.concatenate((er,P-Pd))
        dQ = np.matrix.transpose(J) * dX
        dQ *= alpha
        for i in range(min(len(q), len(dQ.data))):
            q[i] -= dQ.getA()[i][0]
    print("posDif: {}   rotDif: {}".format(posDif, rotDif))
    return radiansToDegrees( q )

#First try to get center of blob, assuming 1 large blob/color we're searching for 
#returns: [x,y], where x and y are pixel coordinates, the center of the largest blob of color targetColor
def getCenterOfBlobFromImage(image = [[[]]], targetColor = [10, 10, 255], maxDeviation = 30, pixelStep = 10, printStats = True):
    #Goal:step over image in grid pattern, and record where the pixel color is close to the target color
    L = []  #List containing all positions in form [x,y] that are close to target color
    avgX = 0
    avgY = 0
    for x in range(0, len(image), pixelStep):
        for y in range(0, len(image[x]), pixelStep):
            pixel = image[x][y]
            #print(pixel)
            if abs(pixel[0] - targetColor[0]) + abs(pixel[1] - targetColor[1]) + abs(pixel[2] - targetColor[2]) < maxDeviation:
                L.append([x,y])
                avgX += x
                avgY += y
    if len(L) < 1:
        if printStats:
            print("Did not find any target.")
        return None, None
    avgX /= len(L)
    avgY /= len(L)
    if printStats:
        print("average x: {} y: {}".format(avgX, avgY))
        print("num pixels found: {}".format(len(L)))

    #Find the distance from each point to the center, and add to point. now L is in form L[pointNum][x=0,y=1,d=2]
    avgD = 0
    for i in range(len(L)):
        d = math.sqrt( math.pow(L[i][0] - avgX,2) + math.pow(L[i][1]-avgY,2) )
        avgD += d
        L[i].append(d)
    avgD /= len(L) 
    print("average distance from center: {}".format(avgD))

    #Compute standard deviation
    stdDev = 0
    for i in range(len(L)):
        stdDev += math.pow(L[i][2] - avgD, 2)
    stdDev = math.sqrt( stdDev/len(L) )
    if printStats: 
        print("stddev of distance from points to center: {}".format(stdDev))

    #Remove all samples which are outside 1 stdDev
    L2 = []
    for i in range(len(L)):
        if avgD + stdDev > L[i][2] and avgD - stdDev < L[i][2]:
            L2.append(L[i])
    L = L2

    #Recompute average, now that we've removed outliers
    avgX = 0
    avgY = 0
    for i in range(len(L)):
        avgX += L[i][0]
        avgY += L[i][1]
    avgX /= len(L)
    avgY /= len(L)
    
    avgX = round(avgX)
    avgY = round(avgY)
    
    if printStats:
        for l in L:
            for x in range(2):
                for y in range(2):
                    #print(image[x+round(l[0])][y+round(l[1])])
                    image[x+round(l[0])][y+round(l[1])][0] = y*250
                    image[x+round(l[0])][y+round(l[1])][1] = y*250
                    image[x+round(l[0])][y+round(l[1])][2] = y*250

        for x in range(-10,10):
            image[avgX][x+avgY][0] = 255
            image[avgX][x+avgY][1] = 255
            image[avgX][x+avgY][2] = 255
            image[x+avgX][avgY][0] = 255
            image[x+avgX][avgY][1] = 255
            image[x+avgX][avgY][2] = 255
        iw = widgets.Image(format='jpg', width=len(image), height=len(image[0]))
        display(iw)
        iw.value = bytes(cv2.imencode('.jpg',image)[1])
        print("final average x: {}  y: {}".format(avgX, avgY))
    return [avgX, avgY]


def displayImage(img):
    iw = widgets.Image(format='jpg', width=len(img), height=len(img[0]))  #This is for displaying the image in Jupyter
    display(iw)
    iw.value = bytes(cv2.imencode('.jpg',img)[1])



######################################################################################################################################################################
######################################################################################################################################################################
##A general class which can be used to communicate with the Dofbot
class Dofbot():
    def __init__(self):
        self.arm = Arm_Device()        #Arm device - object we use to communicate with physical robot
        self.clawPose = 120            #Claw joint angle (fully open = 0, fully closed = 180)
        self.pose = [90,90,90,90,90]   #Joint angles for joints 1-5
        self.camera = None             #This will become populated iff dofbot.getImage() is called
    def goToPose(self, pose = [90,90,90,90,90], moveTime_ms = 1000, sleepWhileMoving = True):
        self.pose = pose.copy()
        self.arm.Arm_serial_servo_write6(self.pose[0],self.pose[1],self.pose[2],self.pose[3],self.pose[4],self.clawPose, moveTime_ms)
        if sleepWhileMoving:
            time.sleep(moveTime_ms/1000)
    def closeClaw(self, angleDeg=None, moveTime_ms = 1000, sleepWhileMoving = True):
        if angleDeg is None:
            angleDeg = 170
        if moveTime_ms is None:
            moveTime_ms = 1000
        self.clawPose = angleDeg
        self.arm.Arm_serial_servo_write(6, self.clawPose, moveTime_ms)
        if sleepWhileMoving:
            time.sleep(moveTime_ms/1000)
    def openClaw(self, angleDeg=None, moveTime_ms = None, sleepWhileMoving = True):
        if angleDeg is None:
            angleDeg = 0
        if moveTime_ms is None:
            moveTime_ms = 1000
        self.clawPose = angleDeg
        self.arm.Arm_serial_servo_write(6, self.clawPose, moveTime_ms)
        if sleepWhileMoving:
            time.sleep(moveTime_ms/1000)
    def getImage(self, displayImage = True):
        #returns image in 3d array form [x][y][blue,green,red]
        for i in range(10):   #Step through. Sometimes the camera port is not 0, and this will essentially just find the first camera available 
            time.sleep(0.25)  #Delaying seems to help... makes finding the camera more consistent
            if self.camera is None:
                self.camera = cv2.VideoCapture(i)
                self.camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
                self.camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
                self.camera.set(5,10)            #This line is in Dofbot's example code, but no explanation is provided.
                time.sleep(0.25)
            ret, image = self.camera.read()      #Read the image from the camera  (Returns ret=[True, False] and image=[x][y][rgb])
            if ret:                                                       
                if displayImage:
                    #iw = widgets.Image(format='jpg', width=len(image), height=len(image[0]))  #This is for displaying the image in Jupyter
                    #display(iw)
                    #iw.value = bytes(cv2.imencode('.jpg',image)[1])                           #Set iw value to byte jpg format of image
                    displayImage(image)
                return image
        self.camera = None
        print("Failed to get image from camera")
        return None
    def destroy(self):
        if self.camera is not None:
            self.camera.release()
    def IBVS(self):
        print("Beginning IBVS")
        
        imagingPoses = [[90,90,70,-30,90],[80,90,70,-30,90],[85,90,70,-30,90],[95,90,70,-30,90],[100,90,70,-30,90]]
        originalImages = []
        bwImages = []
        foundPositions = []
        
        smallestIndex = -1
        smallestDist = 100000000
        
        lower = np.array([60, 10, 10])
        upper = np.array([250, 90, 90])
        
        ########################################################
        #Get image
        index = 0
        for imagingPose in imagingPoses:
            self.goToPose(imagingPose)
            originalImg = self.getImage(False)
            if originalImg is None:
                print("Failed to get image - returned None")
                return
            
            imageWidth = len(originalImg[0])
            imageHeight = len(originalImg)
            
            displayImage(cv2.inRange(originalImg, np.array([0,0,0]), np.array([20,20,20])))
    
            #Blur image, filter, find center of bw filtered image
            #also, find average color. continue if avgColor is too dim, meaning we didn't find a target
            bwImg = cv2.inRange(cv2.blur(cv2.blur(originalImg, (10,10)), (10,10)), lower, upper)
            averageColor = np.average(np.average(bwImg, axis=0), axis=0)
            if averageColor < 1.5:
                continue
            M = cv2.moments(bwImg) 
            if int(M["m00"]) == 0:
                displayImage(bwImg)
                print("Failed to find center of img.")
                return
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        
            #Draw circle at center of blob on original image
            cv2.circle(originalImg, (cx, cy), 5, (255, 255, 255), -1)
            #displayImage(originalImg)
        
            dx = cx - imageWidth/2 - 67
            dy = - (cy - imageHeight/2) + 30
            #print("dx: {}   dy: {}   width: {}  height: {}".format(dx, dy, imageWidth, imageHeight))
            mag = abs(dx**2) + abs(dy**2)
            if mag < smallestDist:
                smallestDist = mag
                smallestIndex = index

            ###########################################################
            Epos, Erot = dofbotFK(self.pose)  #Get end effector position and rotation
            #Ev = Erot * np.matrix([[-1],[0],[0]]) #Get direction vector coming from end effector (Which way it's pointing)
            #print("Current camera direction vector:\n{}".format(Ev))

            #Assume focal length = 800, and default direction is [-1],[0],[0], 
            cameraOffset = np.matrix([[140],[40],[0]])
            directionVector = Erot * normalize(np.matrix([[-800],[dy],[-dx]]))

            #print("Computed Target Vector:\n{}".format(directionVector))

            #Basically, temp = camera position to start. then, we follow "directionVector" until it hits the target plane.
            # Then, we go to position found by temp.
            temp = Epos + Erot * cameraOffset
            runs = 0
            self.openClaw()
            while temp.getA()[2] > 20 and runs < 10000:
                temp += directionVector*2
                runs += 1
            foundPositions.append(temp)
            originalImages.append(originalImg)
            bwImages.append(bwImg)
            index += 1
        
        print("found positions: \n{}\nSmallestIndex: {}".format(foundPositions, smallestIndex))
        q = dofbotIK(foundPositions[smallestIndex], Erot, self.pose.copy())
        self.goToPose(q)
        displayImage(originalImages[smallestIndex])
        displayImage(bwImages[smallestIndex])
        self.closeClaw()
        self.goToPose(imagingPoses[0])
        self.openClaw()

 
######################################################################################################################################################################
######################################################################################################################################################################
# Test functions
def testInverseKinematics():
    poses = [
        [90,90,60,-30,90],
        [90,90,60,0,0],
        [90,30,60,30,90],
        [90,30,0,30,0],
    ]
    for pose in poses:
        pos, rot = dofbotFK(pose)      #compute pos & rot given dofbotFK
        resultQ = dofbotIK(pos, rot)    #compute q given the pos and rot we just found
        pos2, rot2 = dofbotFK(resultQ)  #recompute pos & rot given new q
        posDif = getMagnitude( pos - pos2 ) #Compare previous to new pos and rot
        rotDif = getMagnitude( rot - rot2 )
        print("Input Q -> FK -> pos, rot -> IK  -> output Q: \n{}\n{}".format(pose, [round(q,2) for q in resultQ]))
        print("Pos dif: {:.3f}    rotDif: {:.3f}\n".format(posDif, rotDif))
#testInverseKinematics()



pose1 = [90,90,70,-30,90]  #Imaging pose 1
#pose2 = [90,45,45,30,90]   #Grabbing pose
#pose3 = [90,0,180,-20,0]   #Releasing pose

bot = Dofbot()
#bot.goToPose([90,90,0,90,90])
#bot.goToPose(pose1)
#bot.goToPose([80,90,70,-30,90])
#bot.goToPose([90,80,60,-30,90])
#bot.goToPose([90,80,70,-20,90])
#bot.goToPose([90,80,70,-30,80])
#bot.goToPose(pose1)
bot.IBVS()
#bot.goToPose(pose1)
#bot.goToPose([90,40,60,0,90])
#bot.openClaw()
#bot.goToPose(pose2)
#bot.goToPose(pose3)
bot.destroy()


Beginning IBVS
